# Fashion Retrieval — Evaluation Notebook

Run all 5 evaluation queries and visualise results with score breakdowns.

**Prerequisites:** Index must already be built (`python -m part_a_indexer.index`)


In [ ]:
import sys
sys.path.insert(0, '..')  # project root

from dotenv import load_dotenv
load_dotenv('../.env')

from part_b_retriever.retriever import FashionRetriever

retriever = FashionRetriever(device='cpu')
print('FashionRetriever loaded.')

In [ ]:
EVALUATION_QUERIES = [
    'A person in a bright yellow raincoat',
    'Professional business attire inside a modern office',
    'Someone wearing a blue shirt sitting on a park bench',
    'Casual weekend outfit for a city walk',
    'A red tie and a white shirt in a formal setting',
]
TOP_K = 10

## Query Decomposition Preview

In [ ]:
for i, q in enumerate(EVALUATION_QUERIES, 1):
    components = retriever.get_query_components(q)
    print(f'\nQuery {i}: {q}')
    print(f'  Colors:    {[(c["color"], c["item"]) for c in components["colors"]]}')
    print(f'  Clothing:  {[c["item"] for c in components["clothing"]]}')
    print(f'  Setting:   {components["context"]["setting"]}')
    print(f'  Formality: {components["context"]["formality"]:.2f}')
    print(f'  Style:     {components["style"]}')

## Run All 5 Queries

In [ ]:
import time

all_results = []

for i, query in enumerate(EVALUATION_QUERIES, 1):
    print(f'\n=== Query {i}: {query[:60]} ===')
    t0 = time.time()
    results = retriever.search(query, top_k=TOP_K)
    elapsed = (time.time() - t0) * 1000
    all_results.append(results)

    print(f'  {len(results)} results in {elapsed:.0f}ms')
    for rank, r in enumerate(results[:5], 1):
        print(f'  #{rank} {r.image_id:<30} score={r.overall_score:.3f} '
              f'(S={r.semantic_score:.2f} F={r.fashion_score:.2f} A={r.attribute_score:.2f})')
        print(f'     matched: {r.matching_attributes[:3]}')

## Score Distribution Across Queries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, (results, ax) in enumerate(zip(all_results, axes)):
    scores = [r.overall_score for r in results]
    semantic = [r.semantic_score for r in results]
    fashion = [r.fashion_score for r in results]
    attr = [r.attribute_score for r in results]

    x = range(1, len(results) + 1)
    ax.bar(x, [0.5 * s for s in semantic], label='Semantic (50%)', color='#2ECC71', alpha=0.8)
    ax.bar(x, [0.3 * f for f in fashion], bottom=[0.5 * s for s in semantic],
           label='Fashion (30%)', color='#F39C12', alpha=0.8)
    ax.bar(x, [0.2 * a for a in attr],
           bottom=[0.5 * s + 0.3 * f for s, f in zip(semantic, fashion)],
           label='Attribute (20%)', color='#E74C3C', alpha=0.8)

    ax.set_title(f'Q{i+1}', fontsize=10)
    ax.set_xlabel('Rank')
    ax.set_ylim(0, 1.0)
    if i == 0:
        ax.set_ylabel('Score')
        ax.legend(fontsize=7)

plt.suptitle('Score Breakdown by Query — Top 10 Results', fontsize=13)
plt.tight_layout()
plt.savefig('../results/score_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/score_breakdown.png')